# SkyGuard AI — PPE Detection Training Notebook

**Track:** CV/ML (Padmaja)  
**Purpose:** Download PPE datasets from Roboflow, merge them, and train YOLOv8n for PPE detection.  
**Weeks covered:** Week 1 (#3 dataset sourcing), Week 2 (#14 first-pass training), Week 4 (#32 machinery), Week 5 (#39 hard conditions)

## Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Fill in your `ROBOFLOW_API_KEY` in Cell 3
3. Run cells top-to-bottom — do not skip

## Training phases in this notebook
| Phase | Model name | Epochs | Base weights | Week |
|---|---|---|---|---|
| 1 | `ppe_v1` | 100 | `yolov8n.pt` | 2 |
| 2 | `full_v1` | 80 | `ppe_v1/best.pt` | 4 |
| 3 | `full_v2` | 60 | `full_v1/best.pt` | 5 |

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All outputs will be saved here — create it if it doesn't exist
DRIVE_OUTPUT = '/content/drive/MyDrive/skyguard_ai_cv'
import os
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive mounted. Output directory: {DRIVE_OUTPUT}')

## Cell 2 — Install dependencies

In [ ]:
!pip install ultralytics roboflow -q
print('Installation complete.')

## Cell 3 — GPU check & config

In [ ]:
!nvidia-smi

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Switch to T4 GPU runtime before training.')

# ---- FILL IN YOUR API KEY BELOW ----
ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY_HERE'  # get from app.roboflow.com -> Settings -> API
assert ROBOFLOW_API_KEY != 'YOUR_ROBOFLOW_API_KEY_HERE', 'Set your Roboflow API key first!'

## Cell 4 — Download PPE datasets from Roboflow

We download 3 public datasets and merge them. Class names are remapped to the locked list:
`helmet(0), no-helmet(1), vest(2), no-vest(3), person(4), machinery(5)`

In [ ]:
from roboflow import Roboflow
import os, shutil, yaml
from pathlib import Path

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

RAW_DIR = Path('/content/datasets/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Dataset 1: Hard Hat Workers (Roboflow Universe - widely used PPE dataset)
# Covers: helmet, no-helmet, person
project1 = rf.workspace('roboflow-universe-projects').project('hard-hat-workers')
dataset1  = project1.version(2).download('yolov8', location=str(RAW_DIR / 'hard-hat-workers'))

# Dataset 2: Construction Site Safety
# Covers: helmet, vest, no-helmet, no-vest, person
project2 = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
dataset2  = project2.version(30).download('yolov8', location=str(RAW_DIR / 'construction-site-safety'))

# Dataset 3: PPE Detection (extra vest + machinery coverage)
project3 = rf.workspace('roboflow-universe-projects').project('ppe-detection-bzwpz')
dataset3  = project3.version(1).download('yolov8', location=str(RAW_DIR / 'ppe-detection'))

print('All 3 datasets downloaded.')

# List what we got
for ds_dir in RAW_DIR.iterdir():
    yaml_files = list(ds_dir.glob('*.yaml'))
    if yaml_files:
        with open(yaml_files[0]) as f:
            info = yaml.safe_load(f)
        print(f'  {ds_dir.name}: {info.get("nc","?")} classes → {info.get("names",{})}')

## Cell 5 — Merge datasets + create data.yaml with locked class mapping

In [ ]:
import shutil, yaml
from pathlib import Path

MERGED_DIR = Path('/content/datasets/merged')
for split in ['train', 'val', 'test']:
    (MERGED_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------
# CLASS REMAP TABLE
# Maps each raw dataset's class IDs to the locked SkyGuard class IDs.
# Edit this table after inspecting each downloaded dataset's data.yaml.
# Locked: 0=helmet 1=no-helmet 2=vest 3=no-vest 4=person 5=machinery
# -----------------------------------------------------------------------
LOCKED_CLASSES = ['helmet', 'no-helmet', 'vest', 'no-vest', 'person', 'machinery']
LOCKED_ID = {name: idx for idx, name in enumerate(LOCKED_CLASSES)}

# Per-dataset remap: {raw_class_id: locked_class_id, ...}
# -1 means skip (class not needed in final dataset)
# Update these after running Cell 4 and reading the printed class lists.
DATASET_REMAPS = {
    'hard-hat-workers': {
        # Typical classes: 0=Hardhat, 1=Mask, 2=NO-Hardhat, 3=NO-Mask, 4=NO-Safety Vest,
        #                  5=Person, 6=Safety Cone, 7=Safety Vest, 8=machinery, 9=vehicle
        0: 0,   # Hardhat       → helmet
        1: -1,  # Mask          → skip
        2: 1,   # NO-Hardhat    → no-helmet
        3: -1,  # NO-Mask       → skip
        4: 3,   # NO-Safety Vest→ no-vest
        5: 4,   # Person        → person
        6: -1,  # Safety Cone   → skip
        7: 2,   # Safety Vest   → vest
        8: 5,   # machinery     → machinery
        9: 5,   # vehicle       → machinery
    },
    'construction-site-safety': {
        # Update after seeing printed class names from dataset2
        # Common pattern: 0=Hardhat 1=Mask 2=NO-Hardhat 3=NO-Mask 4=NO-Safety Vest
        #                 5=Person 6=Safety Cone 7=Safety Vest 8=machinery
        0: 0, 1: -1, 2: 1, 3: -1, 4: 3, 5: 4, 6: -1, 7: 2, 8: 5,
    },
    'ppe-detection': {
        # Update after seeing printed class names from dataset3
        # Generic fallback — adjust to match actual class list
        0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5,
    },
}

def remap_label_file(src_path, dst_path, remap):
    """Copy a label file, remapping class IDs. Skip lines for unwanted classes."""
    lines_out = []
    with open(src_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            raw_cls = int(parts[0])
            new_cls = remap.get(raw_cls, -1)
            if new_cls == -1:
                continue
            lines_out.append(f'{new_cls} {" ".join(parts[1:])}')
    if lines_out:
        with open(dst_path, 'w') as f:
            f.write('\n'.join(lines_out) + '\n')

total_images = {'train': 0, 'val': 0, 'test': 0}

for ds_name, remap in DATASET_REMAPS.items():
    ds_dir = RAW_DIR / ds_name
    if not ds_dir.exists():
        print(f'Skipping {ds_name} — not found')
        continue

    for split in ['train', 'valid', 'test']:
        out_split = 'val' if split == 'valid' else split
        img_src = ds_dir / split / 'images'
        lbl_src = ds_dir / split / 'labels'
        if not img_src.exists():
            continue
        for img_path in img_src.glob('*'):
            dst_img = MERGED_DIR / 'images' / out_split / f'{ds_name}_{img_path.name}'
            shutil.copy2(img_path, dst_img)
            lbl_path = lbl_src / img_path.with_suffix('.txt').name
            if lbl_path.exists():
                dst_lbl = MERGED_DIR / 'labels' / out_split / f'{ds_name}_{lbl_path.name}'
                remap_label_file(lbl_path, dst_lbl, remap)
            total_images[out_split] += 1

# Write data.yaml with class-name assertion guard
data_yaml = {
    'path':  str(MERGED_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    6,
    'names': {i: n for i, n in enumerate(LOCKED_CLASSES)},
}
yaml_path = MERGED_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'Merged dataset: train={total_images["train"]} val={total_images["val"]} test={total_images["test"]}')
print(f'Classes: {LOCKED_CLASSES}')
print(f'data.yaml written to: {yaml_path}')

# GUARD: assert the class names are exactly the locked list before any training
expected = set(LOCKED_CLASSES)
with open(yaml_path) as f:
    loaded = yaml.safe_load(f)
actual = set(loaded['names'].values())
assert actual == expected, f'CLASS MISMATCH — training aborted.\nExpected: {expected}\nGot:      {actual}'
print('Class-name guard passed — proceeding to training.')

## Cell 6 — Phase 1: Train ppe_v1 (helmet + vest, Week 2)

~30–45 min on T4. Watch the mAP50 column climb in the output.

In [ ]:
# yolo detect train data=data.yaml model=yolov8n.pt epochs=100 imgsz=640 batch=16 name=ppe_v1
!yolo detect train \
    data=/content/datasets/merged/data.yaml \
    model=yolov8n.pt \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    name=ppe_v1 \
    project=/content/runs \
    patience=15 \
    device=0

## Cell 7 — Validate ppe_v1 + record metrics

In [ ]:
!yolo detect val \
    model=/content/runs/ppe_v1/weights/best.pt \
    data=/content/datasets/merged/data.yaml \
    device=0

# Parse and display the key metrics
import pandas as pd
from pathlib import Path
results_csv = Path('/content/runs/ppe_v1/results.csv')
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    last = df.iloc[-1]
    print(f"\nWeek 2 ppe_v1 — Final Epoch Metrics")
    print(f"  mAP50       : {last.get('metrics/mAP50(B)', 'N/A'):.4f}")
    print(f"  mAP50-95    : {last.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
    print(f"  Precision   : {last.get('metrics/precision(B)', 'N/A'):.4f}")
    print(f"  Recall      : {last.get('metrics/recall(B)', 'N/A'):.4f}")
else:
    print('results.csv not found — check training output above.')

## Cell 8 — Phase 2: Fine-tune full_v1 (add machinery, Week 4)

Starting from ppe_v1/best.pt — only 80 more epochs needed.

In [ ]:
# yolo detect train data=data.yaml model=models/ppe_v1/best.pt epochs=80 name=full_v1
!yolo detect train \
    data=/content/datasets/merged/data.yaml \
    model=/content/runs/ppe_v1/weights/best.pt \
    epochs=80 \
    imgsz=640 \
    batch=16 \
    name=full_v1 \
    project=/content/runs \
    patience=15 \
    device=0

## Cell 9 — Phase 3: Retrain full_v2 with hard-conditions augmentation (Week 5)

Applies brightness jitter, motion blur, mosaic, and random erasing — all via YOLO's
built-in augmentation args. No separate Roboflow augmentation step needed at this scope.

In [ ]:
# yolo detect train data=data.yaml model=full_v1/best.pt epochs=60 name=full_v2
!yolo detect train \
    data=/content/datasets/merged/data.yaml \
    model=/content/runs/full_v1/weights/best.pt \
    epochs=60 \
    imgsz=640 \
    batch=16 \
    name=full_v2 \
    project=/content/runs \
    patience=15 \
    device=0 \
    hsv_h=0.02 \
    hsv_s=0.9 \
    hsv_v=0.6 \
    degrees=5.0 \
    translate=0.1 \
    scale=0.6 \
    mosaic=1.0 \
    erasing=0.5 \
    mixup=0.1

## Cell 10 — Export to ONNX (for Pi 5 deployment, Week 10)

In [ ]:
# Export full_v2/best.pt → ONNX for edge deployment
# Use format=ncnn instead if ONNX is too slow on Pi CPU
!yolo export model=/content/runs/full_v2/weights/best.pt format=onnx simplify=True
print('Export complete. Check /content/runs/full_v2/weights/')

## Cell 11 — Copy all weights to Google Drive

In [ ]:
import shutil
from pathlib import Path

RUNS = Path('/content/runs')
DRIVE_MODELS = Path(DRIVE_OUTPUT) / 'models'

for run_name in ['ppe_v1', 'full_v1', 'full_v2']:
    src_best = RUNS / run_name / 'weights' / 'best.pt'
    dst_dir  = DRIVE_MODELS / run_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    if src_best.exists():
        shutil.copy2(src_best, dst_dir / 'best.pt')
        print(f'Saved: {dst_dir}/best.pt')
    # Also save the full run folder (metrics, plots)
    run_src = RUNS / run_name
    run_dst = DRIVE_MODELS / f'{run_name}_run'
    if run_src.exists() and not run_dst.exists():
        shutil.copytree(run_src, run_dst)
        print(f'Saved run folder: {run_dst}')

# Also save the ONNX export
onnx_src = RUNS / 'full_v2' / 'weights' / 'best.onnx'
if onnx_src.exists():
    shutil.copy2(onnx_src, DRIVE_MODELS / 'full_v2' / 'best.onnx')
    print('Saved ONNX export.')

print(f'\nAll weights saved to Google Drive: {DRIVE_MODELS}')
print('Download best.pt files to cv-model/models/<run_name>/best.pt locally.')